# Curvature of spacetime metrics

Worked examples for `grtensor`, which turns a line element $\mathrm{d}s^2$ written in LaTeX into Christoffel symbols, the Riemann, Ricci and Einstein tensors, the Kretschmann scalar, the geodesic equations and the conserved momenta. Conventions follow Schutz, Carroll and MTW:

$$\Gamma^a_{bc} = \tfrac12 g^{ad}(\partial_b g_{dc} + \partial_c g_{db} - \partial_d g_{bc}),\qquad R^a{}_{bcd} = \partial_c\Gamma^a_{db} - \partial_d\Gamma^a_{cb} + \Gamma^a_{ce}\Gamma^e_{db} - \Gamma^a_{de}\Gamma^e_{cb},$$

$$R_{bd} = R^a{}_{bad},\qquad R = g^{bd}R_{bd},\qquad G_{ab} = R_{ab} - \tfrac12 g_{ab}R.$$

The same computations are available interactively with `streamlit run app.py`.

In [1]:
import time

import sympy as sp
from IPython.display import Math, display

from grtensor import EXAMPLES, Metric, from_line_element, render


def show(lines, empty=None):
    """Display LaTeX lines; `empty` is shown when every component vanishes."""
    for line in lines or [empty]:
        display(Math(line))

## 1. Schwarzschild from its line element

`d\Omega^2` stands for $\mathrm{d}\theta^2 + \sin^2\theta\,\mathrm{d}\phi^2$ in the last two coordinates.

In [2]:
schwarzschild = from_line_element(
    r"ds^2 = -\left(1-\frac{2M}{r}\right)dt^2 + \frac{dr^2}{1-\frac{2M}{r}} + r^2\,d\Omega^2",
    r"t, r, \theta, \phi",
)
show([render.line_element(schwarzschild)])
show(render.christoffel(schwarzschild))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

It is a vacuum solution, and the Kretschmann scalar $48M^2/r^6$ shows that $r = 2M$ is not a curvature singularity.

In [3]:
show(render.ricci(schwarzschild), empty=r"R_{ab} = 0")
show(render.kretschmann(schwarzschild))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 2. A static spherically symmetric star

This is the example of the previous notebook, with two unknown functions $\Phi(r)$ and $\Lambda(r)$. The Einstein tensor reproduces Schutz, *A First Course in General Relativity*, eqs. (10.14)–(10.15) for $G_{tt}$ and $G_{rr}$ (checked in the tests).

In [4]:
ex = EXAMPLES["Static spherically symmetric (Schutz §10.1)"]
star = from_line_element(ex["line_element"], ex["coords"], functions=ex["functions"])
show([render.line_element(star)])
show(render.einstein(star))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 3. Kerr, written as in the report

The line element (1) of the report, with the abbreviations $\Sigma = r^2 + a^2\cos^2\theta$ and $\Delta = r^2 + a^2 - 2Mr$. The metric does not depend on $t$ or $\phi$, so $p_t = -E$ and $p_\phi = L$ are conserved, as in Lemma 3.1 of the report. The last line is the normalization $g_{ab}\dot x^a \dot x^b = \kappa$.

In [5]:
ex = EXAMPLES["Kerr (Boyer–Lindquist, as in the report)"]
kerr = from_line_element(ex["line_element"], ex["coords"], definitions=ex["definitions"])
show(render.conserved(kerr))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [6]:
start = time.perf_counter()
print(f"{len(render.christoffel(kerr))} nonzero Christoffel symbols, "
      f"{len(render.riemann(kerr, lower=True))} independent nonzero R_abcd")
print("Ricci flat:", kerr.ricci == sp.zeros(4))
show(render.kretschmann(kerr))
print(f"{time.perf_counter() - start:.1f} s")

20 nonzero Christoffel symbols, 13 independent nonzero R_abcd
Ricci flat: True


<IPython.core.display.Math object>

8.6 s


## 4. FLRW and the Friedmann equations

With $G_{ab} = 8\pi T_{ab}$ and a perfect fluid, $G_{tt} = 8\pi\rho$ is the first Friedmann equation $(\dot a/a)^2 + k/a^2 = 8\pi\rho/3$.

In [7]:
ex = EXAMPLES["FLRW (curvature k)"]
flrw = from_line_element(ex["line_element"], ex["coords"], functions=ex["functions"])
show(render.einstein(flrw)[:1])
show(render.ricci_scalar(flrw))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 5. A metric given as a matrix

`Metric(coords, g)` takes the components directly. Here is the unit sphere in the coordinates $(\theta, v = \phi + \theta)$, where the metric has an off-diagonal term. The scalar curvature must still be $2$; the previous notebook got $-2\sin^2\theta - 2$ here, because it summed only the diagonal terms $g^{ii}R_{ii}$ and contracted the wrong pair of Riemann indices.

In [8]:
theta, v = sp.symbols("theta v", positive=True)
g = sp.Matrix([[1 + sp.sin(theta)**2, -sp.sin(theta)**2], [-sp.sin(theta)**2, sp.sin(theta)**2]])
sphere = Metric([theta, v], g)
show(render.ricci_scalar(sphere))

<IPython.core.display.Math object>